# CNN Project: Brain Tumor Classification from MRI Scans

In this project I classify brain MRI scans into 4 classes (glioma, meningioma, no tumor, pituitary) using a CNN in PyTorch.

Steps:
1. Load and look at the dataset
2. Train a CNN from scratch and check for overfitting / underfitting
3. Fix the problems I find (augmentation, dropout, batch norm, weight decay)
4. Fine-tune a pretrained ResNet18
5. Compare all the models on a test set, look at the confusion matrix and at Grad-CAM heatmaps that show where each network looks when it decides

**Dataset:** Brain Tumor MRI Dataset (Kaggle) - 7,200 MRI scans, downloaded automatically with kagglehub .

In [ ]:
import pathlib
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import kagglehub

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
root = pathlib.Path(kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset"))
train_dir, test_dir = root / "Training", root / "Testing"

raw_data = datasets.ImageFolder(train_dir)   # no transform, just to look at the data
class_names = raw_data.classes

counts = {c: len(list((train_dir / c).glob("*.*"))) + len(list((test_dir / c).glob("*.*")))
          for c in class_names}
print("Classes:", class_names)
print("Images per class:", counts)
print("Total images:", sum(counts.values()))

In [ ]:
plt.figure(figsize=(9, 7))
for i, name in enumerate(class_names):
    idx = raw_data.targets.index(i)
    img, label = raw_data[idx]
    plt.subplot(2, 2, i + 1)
    plt.imshow(img.convert("RGB"))
    plt.title(name)
    plt.axis("off")
plt.show()

In [ ]:
IMG_SIZE = 224

normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats, needed for the pretrained model

basic_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize
])

aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    normalize
])

data_basic = ConcatDataset([
    datasets.ImageFolder(train_dir, transform=basic_transform),
    datasets.ImageFolder(test_dir, transform=basic_transform),
])
data_aug = ConcatDataset([
    datasets.ImageFolder(train_dir, transform=aug_transform),
    datasets.ImageFolder(test_dir, transform=aug_transform),
])

indices = torch.randperm(len(data_basic)).tolist()
n_train = int(0.70 * len(indices))
n_val = int(0.15 * len(indices))
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

train_loader_basic = DataLoader(Subset(data_basic, train_idx), batch_size=64, shuffle=True, num_workers=2)
train_loader_aug = DataLoader(Subset(data_aug, train_idx), batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(Subset(data_basic, val_idx), batch_size=64, num_workers=2)
test_loader = DataLoader(Subset(data_basic, test_idx), batch_size=64, num_workers=2)

print("Train:", len(train_idx), "Val:", len(val_idx), "Test:", len(test_idx))

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, weight_decay=0):
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    # only train the parameters that are not frozen
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=lr, weight_decay=weight_decay)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        # ---- training ----
        model.train()
        total_loss, correct, total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss = total_loss / total
        train_acc = correct / total

        # ---- validation ----
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = loss_fn(outputs, labels)
                total_loss += loss.item() * images.size(0)
                correct += (outputs.argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_loss = total_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} - train loss: {train_loss:.3f}, train acc: {train_acc:.3f}, "
              f"val loss: {val_loss:.3f}, val acc: {val_acc:.3f}")

    return history


def test_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total


def plot_history(history, title):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="train")
    plt.plot(history["val_loss"], label="validation")
    plt.title(title + " - Loss")
    plt.xlabel("Epoch")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="train")
    plt.plot(history["val_acc"], label="validation")
    plt.title(title + " - Accuracy")
    plt.xlabel("Epoch")
    plt.legend()
    plt.show()


def check_fit(history):
    # compare the last train accuracy with the last validation accuracy
    train_acc = history["train_acc"][-1]
    val_acc = history["val_acc"][-1]
    gap = train_acc - val_acc
    print(f"Final train acc: {train_acc:.3f}, final val acc: {val_acc:.3f}, gap: {gap:.3f}")
    if train_acc < 0.80:
        print("-> Underfitting (the model is not even learning the training data well)")
    elif gap > 0.05:
        print("-> Overfitting (train accuracy is much higher than validation accuracy)")
    else:
        print("-> Looks like a good fit")

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 224 -> 112
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 112 -> 56
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)   # 56 -> 28
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 256),
            nn.ReLU(),
            nn.Linear(256, len(class_names))
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


model1 = SimpleCNN()
history1 = train_model(model1, train_loader_basic, val_loader, epochs=15, lr=0.001)

In [ ]:
plot_history(history1, "Model 1 (simple CNN)")
check_fit(history1)

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout(0.1),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout(0.2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, len(class_names))
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


model2 = ImprovedCNN()
history2 = train_model(model2, train_loader_aug, val_loader, epochs=25, lr=0.001, weight_decay=1e-4)

In [ ]:
plot_history(history2, "Model 2 (improved CNN)")
check_fit(history2)

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in resnet.parameters():
    param.requires_grad = False

resnet.fc = nn.Linear(resnet.fc.in_features, len(class_names))

history_step1 = train_model(resnet, train_loader_aug, val_loader, epochs=5, lr=0.001)

In [ ]:
for param in resnet.parameters():
    param.requires_grad = True

history_step2 = train_model(resnet, train_loader_aug, val_loader, epochs=8, lr=0.0001, weight_decay=1e-4)

history3 = {}
for key in history_step1:
    history3[key] = history_step1[key] + history_step2[key]

In [ ]:
plot_history(history3, "Model 3 (fine-tuned ResNet18)")
check_fit(history3)

In [ ]:
acc1 = test_model(model1, test_loader)
acc2 = test_model(model2, test_loader)
acc3 = test_model(resnet, test_loader)

print("Model 1 (simple CNN, scratch):   ", round(acc1 * 100, 2), "%")
print("Model 2 (improved CNN, scratch): ", round(acc2 * 100, 2), "%")
print("Model 3 (fine-tuned ResNet18):   ", round(acc3 * 100, 2), "%")

results = {"Model 1 (simple CNN)": acc1,
           "Model 2 (improved CNN)": acc2,
           "Model 3 (fine-tuned ResNet18)": acc3}
best_name = max(results, key=results.get)
best_model = {"Model 1 (simple CNN)": model1,
              "Model 2 (improved CNN)": model2,
              "Model 3 (fine-tuned ResNet18)": resnet}[best_name]
print("Best model:", best_name)

In [ ]:
names = ["Model 1\nsimple CNN", "Model 2\nimproved CNN", "Model 3\nfine-tuned ResNet18"]
accs = [acc1 * 100, acc2 * 100, acc3 * 100]

plt.figure(figsize=(7, 4))
plt.bar(names, accs, color=["tomato", "orange", "seagreen"])
plt.ylabel("Test accuracy (%)")
plt.ylim(min(accs) - 10, 100)
for i, a in enumerate(accs):
    plt.text(i, a + 0.5, f"{a:.1f}%", ha="center")
plt.title("Test accuracy comparison")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history1["val_acc"], label="Model 1 (simple CNN)")
plt.plot(history2["val_acc"], label="Model 2 (improved CNN)")
plt.plot(history3["val_acc"], label="Model 3 (fine-tuned ResNet18)")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Validation accuracy of all models")
plt.legend()
plt.show()

In [ ]:
best_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = best_model(images.to(device))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=90)
plt.yticks(range(len(class_names)), class_names)
for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=9)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion matrix - {best_name}")
plt.show()

print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
def get_target_layer(model):
    # the layer whose output is the last convolutional feature map
    if isinstance(model, models.ResNet):
        return model.layer4
    return [m for m in model.conv_layers if isinstance(m, nn.Conv2d)][-1]


def make_gradcam_heatmap(img, model, target_layer):
    # Grad-CAM for one image: gradient of the class score w.r.t. the last conv feature map,
    # averaged over channels, used to weight that feature map (red = important, blue = ignored)
    saved = {}

    def hook(module, inp, out):
        out.retain_grad()
        saved["act"] = out

    handle = target_layer.register_forward_hook(hook)
    model.zero_grad()
    output = model(img.to(device))
    pred = int(output.argmax(1).item())
    output[0, pred].backward()
    handle.remove()

    act = saved["act"].detach().cpu().numpy()[0]    # (C, h, w)
    grad = saved["act"].grad.cpu().numpy()[0]       # (C, h, w)
    weights = grad.mean(axis=(1, 2))                # one importance weight per channel
    cam = np.maximum((weights[:, None, None] * act).sum(axis=0), 0)
    cam = cam / (cam.max() + 1e-8)
    return cam, pred


def unnormalize(t):
    # tensor (3, H, W) -> numpy (H, W, 3) in [0, 1] for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = t.cpu().numpy().transpose(1, 2, 0) * std + mean
    return np.clip(img, 0, 1)


def overlay_heatmap(img, cam, alpha=0.4):
    # resize the heatmap over the image and blend it (jet colors)
    cam = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((img.shape[1], img.shape[0]))) / 255.0
    heat = plt.cm.jet(cam)[..., :3]
    return np.clip(heat * alpha + img * (1 - alpha), 0, 1)

In [ ]:
images, labels = next(iter(test_loader))

plt.figure(figsize=(12, 8))
for i in range(6):
    cam, pred = make_gradcam_heatmap(images[i:i+1], best_model, get_target_layer(best_model))
    plt.subplot(2, 3, i + 1)
    plt.imshow(overlay_heatmap(unnormalize(images[i]), cam))
    correct = pred == labels[i].item()
    plt.title(f"pred: {class_names[pred]} | true: {class_names[labels[i]]}",
              color="green" if correct else "red")
    plt.axis("off")
plt.suptitle("Grad-CAM: where does the best model look?")
plt.show()

In [ ]:
img = images[0:1]

cam2, pred2 = make_gradcam_heatmap(img, model2, get_target_layer(model2))
cam3, pred3 = make_gradcam_heatmap(img, resnet, get_target_layer(resnet))

plt.figure(figsize=(11, 4))
plt.subplot(1, 3, 1)
plt.imshow(unnormalize(images[0]))
plt.title("original")
plt.axis("off")
plt.subplot(1, 3, 2)
plt.imshow(overlay_heatmap(unnormalize(images[0]), cam2))
plt.title(f"Model 2 (improved CNN)\npred: {class_names[pred2]}")
plt.axis("off")
plt.subplot(1, 3, 3)
plt.imshow(overlay_heatmap(unnormalize(images[0]), cam3))
plt.title(f"Model 3 (fine-tuned ResNet18)\npred: {class_names[pred3]}")
plt.axis("off")
plt.suptitle("Where does each model look?")
plt.show()